# Лабораторная работа №1: Разведочный анализ данных (EDA)
**Задача:** Регрессия (прогнозирование стоимости автомобиля `Selling_Price`).

**Цель:** Очистить данные, сформировать новые информативные признаки, визуализировать ключевые зависимости и подготовить финальный датасет.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


# 1. Загрузка датасета
# Убедитесь, что имя файла совпадает с тем, как он сохранен в папке data/ (например, car data.csv или dataset.csv)
df = pd.read_csv('../data/car data.csv')

# Приведем названия колонок к единому удобному виду без случайных пробелов
df.columns = df.columns.str.strip()

print("Форма таблицы (строк, столбцов):", df.shape)
df.info()
display(df.head())
display(df.describe())

# Определение типов признаков
target_col = 'Selling_Price'
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop(target_col).tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("\nЦелевая переменная:", target_col)
print("Числовые признаки:", numeric_cols)
print("Категориальные признаки:", categorical_cols)

**Вывод по первичному знакомству:**

В датасете 301 строка и 9 столбцов. Пропусков в типах данных на первый взгляд нет. Целевая переменная `Selling_Price` является непрерывной числовой величиной. Присутствуют категориальные признаки: тип топлива, тип продавца, трансмиссия и название автомобиля.

In [ ]:
# 1. Проверка и удаление дубликатов
initial_len = len(df)
df = df.drop_duplicates()
print(f"Удалено дубликатов: {initial_len - len(df)}")

# 2. Проверка пропусков
null_counts = df.isnull().sum()
print("\nКоличество пропусков по колонкам:\n", null_counts)

# 3. Приведение типов данных
# Столбец Owner логически является категорией (количество владельцев 0, 1, 3)
if 'Owner' in df.columns and 'Owner' in numeric_cols:
    numeric_cols.remove('Owner')
    categorical_cols.append('Owner')

# 4. Feature Engineering (Создание новых признаков)
# Признак 1: Возраст автомобиля на текущий момент (2026 год)
current_year = 2026
df['Car_Age'] = current_year - df['Year']
numeric_cols.append('Car_Age')

# Признак 2: Отношение цены продажи к исходной цене автосалона (процент сохранения стоимости)
if 'Present_Price' in df.columns:
    df['Price_Retention_Ratio'] = df['Selling_Price'] / (df['Present_Price'] + 1e-5)
    numeric_cols.append('Price_Retention_Ratio')

print("\nДанные успешно очищены. Новые признаки: 'Car_Age', 'Price_Retention_Ratio'")
display(df.head())

**Вывод по этапу очистки:**

* Удалены обнаруженные строки-дубликаты.
* Пропущенные значения отсутствуют.
* Столбец `Owner` переквалифицирован в категориальный признак.
* Сгенерирован признак `Car_Age` (возраст автомобиля), так как для моделей регрессии возраст линейнее отражает амортизацию, чем абсолютный год выпуска.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Selling_Price'], kde=True, bins=25, color='royalblue')
plt.title('Распределение целевой переменной (Selling_Price)')
plt.xlabel('Цена продажи (в лакхах)')
plt.ylabel('Плотность / Частота')
plt.tight_layout()
plt.savefig('graph1.png')
plt.show()

**Вывод по графику 1:**

Распределение целевой переменной скошено вправо: большинство автомобилей продается по цене до 10 лакхов, при этом есть небольшой хвост дорогих премиальных автомобилей. В дальнейшем обучении модели может потребоваться логарифмирование целевой переменной.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='Present_Price', y='Selling_Price', hue='Fuel_Type', palette='tab10', alpha=0.8)
plt.title('Зависимость Selling_Price от Present_Price')
plt.xlabel('Текущая цена новой машины (Present_Price)')
plt.ylabel('Цена продажи (Selling_Price)')
plt.tight_layout()
plt.savefig('graph2.png')
plt.show()

**Вывод по графику 2:**

Наблюдается выраженная прямая линейная зависимость: чем выше базовая стоимость автомобиля из салона, тем выше вторичная цена продажи. Дизельные автомобили в среднем имеют более высокую стоимость по сравнению с бензиновыми.

In [ ]:
plt.figure(figsize=(8, 4))
seller_col = 'Selling_type' if 'Selling_type' in df.columns else 'Seller_Type'
sns.boxplot(data=df, x=seller_col, y='Selling_Price', hue='Transmission', palette='Set2')
plt.title('Распределение цены продажи по продавцам и типам КПП')
plt.xlabel('Тип продавца')
plt.ylabel('Цена продажи (Selling_Price)')
plt.tight_layout()
plt.savefig('graph3.png')
plt.show()


**Вывод по графику 3:**

Автомобили, продаваемые дилерами, стоят заметно дороже, чем машины от частных лиц (`Individual`). Машины с автоматической коробкой передач в среднем имеют больший разброс цен и более высокую медианную стоимость.

In [ ]:
plt.figure(figsize=(8, 6))
corr_matrix = df.corr(numeric_only=True)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Матрица линейных корреляций')
plt.tight_layout()
plt.savefig('graph4.png')
plt.show()

**Вывод по графику 4:**

Наибольшую корреляцию с целевой переменной демонстрирует признак `Present_Price` (около 0.88). Признак `Car_Age` демонстрирует отрицательную корреляцию: с увеличением возраста цена автомобиля закономерно снижается.

In [ ]:
kms_col = 'Driven_kms' if 'Driven_kms' in df.columns else 'Kms_Driven'

fig = px.scatter(
    df,
    x='Car_Age',
    y='Selling_Price',
    size=kms_col,
    color='Fuel_Type',
    hover_name='Car_Name',
    title='Интерактивный анализ: Возраст vs Цена (размер маркера — пробег)'
)

# Сохраняем интерактивный файл в папку eda
fig.write_html('graph5.html')
fig.show()

**Вывод по графику 5:**

При интерактивном наведении подтверждается, что наиболее дорогие автомобили сосредоточены в возрастном диапазоне до 10–12 лет. Машины с высоким пробегом преимущественно относятся к бензиновым или дизельным седанам среднего ценового сегмента.

In [ ]:
# Сохраняем обработанные данные в формате pickle в папку data/
df.to_pickle('../data/clean_dataset.pkl')
print("Очищенный датасет успешно сохранен в ../data/clean_dataset.pkl")

## Итоговые выводы

1. **Очистка данных:** Удалены строки-дубликаты, пропусков в значениях не обнаружено, признак `Owner` преобразован к категориальному типу.
2. **Новые признаки:** Сформированы переменные `Car_Age` (возраст автомобиля) и `Price_Retention_Ratio`.
3. **Закономерности:**
   * Цена автомобиля линейно связана с исходной салонной ценой (`Present_Price`).
   * Возраст и пробег оказывают негативное влияние на стоимость.
   * Автомобили у дилеров и с автоматической коробкой передач оцениваются рынком выше.